## For an analysis between different dataframes, where for example inconsistencies might arise, look at merges.ipynb

### Import modules and data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

format = '%Y-%m-%d'

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [ ]:
water_path = PATHS['pre_EDA'] / 'water_for_EDA.csv'
water_pd = pd.read_csv(water_path)
water_pd.head()

In [ ]:
water_pd.sort_values(by=['id', 'date'], ignore_index=True, inplace=True)


### Basic analysis first

In [ ]:
water_pd['date'] = pd.to_datetime(water_pd['date'], format=format)
water_pd.info()

In [ ]:
water_pd.duplicated().sum()

So for this dataset there are no repeated rows

In [ ]:
water_pd.describe()

- There are no negative storage values, which is great.
- ids aren't consecutive

### Missing Values Analysis

In [ ]:
water_pd.isna().sum()

As there are only two storage missing values, and we will deal afterwards with missing dates, we can drop these rows for now.

In [ ]:
water_pd.dropna(inplace=True)

### Visualization of the years we have in the whole dataframe

In [ ]:
# Getting years with the last two digits format
years = water_pd['date'].dt.year.astype(str).str[-2:]
year_counts = years.value_counts().sort_index()

In [ ]:
# Seaborn barplot to visualize the count of different years
plt.figure(figsize=(10, 6))
sns.barplot(x=year_counts.index, y=year_counts.values, hue=year_counts.index, palette='viridis', legend=False)
plt.title('Count of Different Years in Water Data')
plt.xlabel('Year')
plt.ylabel('Count')
plt.tight_layout()

### Dates Missing:

In [ ]:
df = water_pd.copy()
df['date_difference'] = df.groupby('id')['date'].diff()
df['date_difference'].value_counts()

There are two main groups:
- Until 84 days of lack: they can be managed with a bbfill
- More than 1000 days lack: those cases need to be studied further, but they look like an imputation

In [ ]:
df[df['date_difference'] > pd.Timedelta(days=7)]

Now we have located which reservoirs have these lacks

In [ ]:
df['date_difference'].value_counts().index.sort_values()

In [ ]:
mask = df['date_difference'].isin(df['date_difference'].value_counts().index.sort_values()[-2:])
df[mask]['id'].unique()

So these are the two reservoirs that need to be studied further (376 and 396)

In [ ]:
def plot_reservoir(reservoir_id):
    df_reservoir = df_full[df_full['id'] == reservoir_id]
    plt.figure(figsize=(10, 5))
    plt.plot(df_reservoir['date'], df_reservoir['storage'], label='Water Level')
    plt.title(f'Reservoir {reservoir_id} Water Level Over Time')
    plt.xlabel('Date')
    plt.ylabel('Water Level')
    plt.legend()
    plt.show()

In [ ]:
for reservoir in df[mask]['id'].unique():
    plot_reservoir(reservoir)

They have a big gap because of certain outliers that will be removed, so that this gap doesn't exist anymore. These aren't the only reservoirs that don't start in 1988-01-05, as we can see in the next cell:

In [ ]:
print(f"There are {df['id'].nunique()} reservoirs in the dataset.")
print(f"There are {df['date'].eq('1988-01-05').sum()} reservoirs that start in 1988-01-05.")


### Studying median storage from all reservoirs per every week of the year

In [ ]:
cleaned_water_path = PATHS['cleaned_data_notebooks'] / 'water_cleaned.parquet'
df = pd.read_parquet(cleaned_water_path)

In [ ]:
df['week_idx'] = df['date'].dt.isocalendar().week
df

Barplot of the median storage from all reservoirs per every week of the year:

In [ ]:
# Barplot of the median storage from all reservoirs per every week of the year

weekly_median = df.groupby('week_idx')['storage'].median()
plt.bar(weekly_median.index, weekly_median.values)
plt.xlabel('Week Index')
plt.ylabel('Median Storage')
plt.title('Median Storage from all Reservoirs per Week')
plt.show()

So it seems that there is a clear pattern in the data, with certain weeks showing consistently higher or lower median storage levels across all reservoirs

### Feature Engineering Proposal

- We have witnessed yearly seasonality, so apart from the current storage, we could also include multiples of 52-weeks lags
- The last years' storage levels are also essential for predicting next year's data, rather than the further ones (they will likely be taken apart)
- We could also explore interaction features, such as the combination of week index and year, to capture any potential trends over time
- Rolling statistics could be beneficial, such as the rolling mean or median over the last few weeks to smooth out short-term fluctuations and highlight longer-term trends
- It could be useful to try, instead of just storage, to use a feature that shows its ratio over the capacity, which is registered in reservoirs dataset